# 13 — Alternative ML Targets: 1R and 2R Winners

Önceki `Meta_Label = Net_Return > 0` modeli, Validation portföyünde
orijinal Robot'u geçemedi. Bu notebook trend-following sistemine daha uygun
iki hedefi araştırır:

- **Meta_Label_1R:** İşlem en az 1R kazandırdı mı?
- **Big_Winner_Label_2R:** İşlem en az 2R kazandırdı mı?

Amaç kazanma oranını yükseltmek değil; Robot'un az sayıdaki büyük trend
işlemlerini koruyup düşük kaliteli sinyalleri elemek.

Metodoloji:

1. Model seçimi yalnızca Development purged cross-validation ile yapılır.
2. Filtre/eşik seçimi yalnızca Validation portföy backtestiyle yapılır.
3. Hiçbir aday katı kabul şartlarını sağlamazsa final sistem Baseline Robot kalır.
4. Audit dönemi yalnızca Validation kararı verildikten sonra raporlanır.
5. Audit dönemi daha önce görüldüğü için gerçek anlamda dokunulmamış test değildir.


In [1]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
from joblib import dump

PROJECT_ROOT = (
    Path.cwd()
    if (Path.cwd() / "src").exists()
    else Path.cwd().parent
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

%load_ext autoreload
%autoreload 2

from src.features import add_indicators
from src.signals import (
    build_market_regime,
    add_robot_scores,
)
from src.presets import (
    FINAL_STRATEGY_CONFIG,
    FINAL_PORTFOLIO_CONFIG,
)
from src.ml_dataset import (
    BASE_FEATURE_COLUMNS,
    add_meta_features,
)
from src.ml_training import (
    build_candidate_models,
    create_purged_expanding_folds,
    folds_summary,
    cross_validate_models,
    train_on_development_predict_validation,
)
from src.ml_portfolio import (
    MLFilterConfig,
    add_model_probabilities,
    run_filter_grid,
    add_baseline_differences,
    validation_acceptance_table,
)
from src.ml_targets import (
    add_alternative_targets,
    target_diagnostics,
    summarize_cv_lift,
    select_cv_champion,
    target_threshold_table,
    build_validation_filter_grid,
    combine_acceptance_results,
    select_global_validation_champion,
)
from src.benchmark import build_benchmark_equity
from src.metrics import portfolio_metrics


## 1. Olay veri setini ve temiz fiyatları yükle


In [2]:
events = pd.read_parquet(
    PROJECT_ROOT
    / "results"
    / "ml"
    / "robot_meta_label_training.parquet"
)

for column in [
    "Signal_Date",
    "Entry_Date",
    "Exit_Date",
]:
    events[column] = pd.to_datetime(events[column])

events = add_alternative_targets(events)

stock_prices = pd.read_parquet(
    PROJECT_ROOT
    / "data"
    / "processed"
    / "bist100_robot_clean.parquet"
)

market_prices = pd.read_parquet(
    PROJECT_ROOT
    / "data"
    / "processed"
    / "xu100_robot_clean.parquet"
)

TARGET_COLUMNS = [
    "Meta_Label_1R",
    "Big_Winner_Label_2R",
]

display(
    target_diagnostics(
        dataset=events,
        target_columns=TARGET_COLUMNS,
    )
)


,Period,Event_Count,Positive_Count,Positive_Rate,Average_Return_,Median_Return_,Average_R_Multiple,Target,Positive_Rate_%
0,Audit_2025_Plus,930,193,0.207527,1.620498,-5.144132,0.210730,Meta_Label_1R,20.752688
1,Development,1851,596,0.321988,6.678966,-3.686008,1.140254,Meta_Label_1R,32.198811
2,Validation,1098,289,0.263206,4.325730,-5.017817,0.627123,Meta_Label_1R,26.320583
3,Audit_2025_Plus,930,126,0.135484,1.620498,-5.144132,0.210730,Big_Winner_Label_2R,13.548387
4,Development,1851,435,0.235008,6.678966,-3.686008,1.140254,Big_Winner_Label_2R,23.500810
5,Validation,1098,191,0.173953,4.325730,-5.017817,0.627123,Big_Winner_Label_2R,17.395264


## 2. Günlük Robot özelliklerini hazırla


In [3]:
stock_features = add_indicators(stock_prices)
market_features = add_indicators(market_prices)
market_regime = build_market_regime(market_features)

scored_prices = add_robot_scores(
    stock_features=stock_features,
    market_regime=market_regime,
    config=FINAL_STRATEGY_CONFIG,
    include_reasons=False,
)

featured_prices = add_meta_features(
    scored_prices=scored_prices,
    market_features=market_features,
)

VALIDATION_START = "2023-01-01"
VALIDATION_END = "2024-12-31"
AUDIT_START = "2025-01-01"
AUDIT_END = min(
    featured_prices["Date"].max(),
    market_prices["Date"].max(),
).strftime("%Y-%m-%d")

print("Validation:", VALIDATION_START, "→", VALIDATION_END)
print("Audit:", AUDIT_START, "→", AUDIT_END)


Validation: 2023-01-01 → 2024-12-31
Audit: 2025-01-01 → 2026-07-23


## 3. Her hedef için Development model seçimi ve Validation portföy testi

Bu bölüm birkaç dakika sürebilir. Her hedefte:

- 4 purged expanding fold
- 3 aday model
- Validation olasılık quantile filtreleri
- Günlük Top %75 / %50 / %30 filtreleri

çalıştırılır.


In [4]:
candidate_model_templates = build_candidate_models(
    feature_columns=BASE_FEATURE_COLUMNS,
    random_state=42,
)

target_artifacts = {}
acceptance_frames = []

for target_column in TARGET_COLUMNS:
    print("=" * 90)
    print("TARGET:", target_column)

    development = (
        events.loc[
            events["Period"].eq("Development")
        ]
        .sort_values(["Signal_Date", "Ticker"])
        .reset_index(drop=True)
    )

    validation = (
        events.loc[
            events["Period"].eq("Validation")
        ]
        .sort_values(["Signal_Date", "Ticker"])
        .reset_index(drop=True)
    )

    folds = create_purged_expanding_folds(
        development_data=development,
        n_splits=4,
        initial_train_fraction=0.40,
        embargo_days=5,
    )

    fold_table = folds_summary(
        data=development,
        folds=folds,
        target_column=target_column,
    )

    if not fold_table["Purged_Correctly"].all():
        raise RuntimeError(
            f"Purged fold kontrolü başarısız: {target_column}"
        )

    cv_fold_metrics, cv_predictions = (
        cross_validate_models(
            development_data=development,
            feature_columns=BASE_FEATURE_COLUMNS,
            target_column=target_column,
            models=candidate_model_templates,
            folds=folds,
        )
    )

    cv_lift_summary = summarize_cv_lift(
        cv_fold_metrics
    )

    cv_champion_name = select_cv_champion(
        cv_lift_summary
    )

    print("Development CV champion:", cv_champion_name)
    display(cv_lift_summary)
    display(fold_table)

    selected_template = {
        cv_champion_name: candidate_model_templates[
            cv_champion_name
        ]
    }

    (
        fitted_models,
        validation_event_metrics,
        validation_event_predictions,
    ) = train_on_development_predict_validation(
        development_data=development,
        validation_data=validation,
        feature_columns=BASE_FEATURE_COLUMNS,
        target_column=target_column,
        models=selected_template,
        cutoff_date=VALIDATION_START,
        embargo_days=5,
    )

    validation_model = fitted_models[
        cv_champion_name
    ]

    champion_event_predictions = (
        validation_event_predictions.loc[
            validation_event_predictions["Model"].eq(
                cv_champion_name
            )
        ]
        .copy()
        .reset_index(drop=True)
    )

    event_thresholds = target_threshold_table(
        predictions=champion_event_predictions,
        target_column=target_column,
    )

    validation_probability_prices = (
        add_model_probabilities(
            featured_prices=featured_prices,
            fitted_model=validation_model,
            feature_columns=BASE_FEATURE_COLUMNS,
            start=VALIDATION_START,
            end=VALIDATION_END,
        )
    )

    filter_grid, quantile_table = (
        build_validation_filter_grid(
            validation_probability_prices
        )
    )

    validation_portfolio_results, portfolio_outputs = (
        run_filter_grid(
            probability_prices=validation_probability_prices,
            filters=filter_grid,
            strategy_config=FINAL_STRATEGY_CONFIG,
            portfolio_config=FINAL_PORTFOLIO_CONFIG,
            start=VALIDATION_START,
            end=VALIDATION_END,
        )
    )

    validation_portfolio_results = (
        add_baseline_differences(
            validation_portfolio_results
        )
    )

    acceptance_table = validation_acceptance_table(
        validation_results=validation_portfolio_results,
        baseline_name="Baseline_Robot",
        minimum_trade_fraction=0.50,
        maximum_drawdown_deterioration_pp=3.0,
    )

    acceptance_table["Target"] = target_column
    acceptance_table["Model"] = cv_champion_name
    acceptance_frames.append(acceptance_table)

    print("VALIDATION EVENT METRİKLERİ")
    display(validation_event_metrics)

    print("OLASILIK QUANTILE EŞİKLERİ")
    display(quantile_table)

    print("VALIDATION PORTFÖY SONUÇLARI")
    display(
        validation_portfolio_results[
            [
                "Filter_Name",
                "CAGR_%",
                "Max_Drawdown_%",
                "Profit_Factor",
                "Sharpe",
                "Calmar",
                "Trade_Count",
                "Signal_Pass_Rate_%",
            ]
        ].sort_values(
            ["Calmar", "CAGR_%"],
            ascending=False,
        )
    )

    print("KABUL TABLOSU")
    display(
        acceptance_table[
            [
                "Target",
                "Model",
                "Filter_Name",
                "CAGR_%",
                "Max_Drawdown_%",
                "Profit_Factor",
                "Sharpe",
                "Calmar",
                "Trade_Count",
                "Acceptance_Count",
                "ML_Accepted",
            ]
        ]
    )

    target_artifacts[target_column] = {
        "development": development,
        "validation": validation,
        "fold_table": fold_table,
        "cv_fold_metrics": cv_fold_metrics,
        "cv_lift_summary": cv_lift_summary,
        "cv_champion_name": cv_champion_name,
        "validation_model": validation_model,
        "validation_event_metrics": (
            validation_event_metrics
        ),
        "validation_event_predictions": (
            champion_event_predictions
        ),
        "event_thresholds": event_thresholds,
        "validation_probability_prices": (
            validation_probability_prices
        ),
        "filter_grid": filter_grid,
        "quantile_table": quantile_table,
        "validation_portfolio_results": (
            validation_portfolio_results
        ),
        "acceptance_table": acceptance_table,
        "portfolio_outputs": portfolio_outputs,
    }


TARGET: Meta_Label_1R
Development CV champion: RandomForest


,Model,PR_AUC_Mean,PR_AUC_Min,PR_AUC_Lift_Mean,PR_AUC_Lift_Min,ROC_AUC_Mean,ROC_AUC_Min,Brier_Mean,Brier_Max,Log_Loss_Mean,Fold_Count
0,RandomForest,0.400331,0.222702,0.062678,-0.034340,0.542304,0.417971,0.235270,0.293977,0.666766,4
1,HistGradientBoosting,0.375301,0.227887,0.037648,-0.029155,0.542729,0.446082,0.280475,0.414785,0.917332,4
2,LogisticRegression,0.375172,0.242464,0.037519,0.000359,0.582616,0.537708,0.249433,0.280615,0.691936,4


,Fold,Train_Count,Validation_Count,Train_Positive_Rate_%,Validation_Positive_Rate_%,Train_Signal_End,Train_Exit_End,Validation_Start,Validation_End,Purged_Correctly
0,1,671,285,28.017884,24.210526,2020-10-05,2020-10-08,2020-10-19,2021-04-26,True
1,2,1006,239,27.833002,35.146444,2021-04-19,2021-04-21,2021-04-27,2021-12-10,True
2,3,1192,284,27.265101,25.704225,2021-11-26,2021-12-03,2021-12-13,2022-06-23,True
3,4,1501,314,27.914724,50.000000,2022-06-10,2022-06-17,2022-06-24,2022-12-30,True


VALIDATION EVENT METRİKLERİ


,Positive_Rate_%,Predicted_Positive_Rate_%,PR_AUC,Brier,Log_Loss,Precision_0_50,Recall_0_50,F1_0_50,ROC_AUC,Model,Train_Count,Validation_Count,Train_Positive_Rate_%
0,26.320583,35.883424,0.26045,0.237497,0.666197,0.248731,0.3391,0.286969,0.516568,RandomForest,1777,1098,31.457513


...

,Quantile,Threshold
0,0.25,0.404881
1,0.40,0.511306
2,0.50,0.570413
3,0.60,0.629356
4,0.75,0.728867


VALIDATION PORTFÖY SONUÇLARI


,Filter_Name,CAGR_%,Max_Drawdown_%,Profit_Factor,Sharpe,Calmar,Trade_Count,Signal_Pass_Rate_%
2,Quantile_Q40,38.377131,-17.322586,2.052734,1.860019,2.215439,152,59.747843
1,Quantile_Q25,50.996917,-24.454761,2.129541,2.151742,2.085357,185,74.678169
3,Quantile_Q50,33.449836,-19.503247,1.916607,1.791944,1.715091,147,49.781022
0,Baseline_Robot,36.064262,-23.126455,1.687332,1.575621,1.559438,214,100.000000
4,Quantile_Q60,28.877885,-19.494472,1.855069,1.630278,1.481337,142,39.827472
7,Daily_Top_50pct,30.975798,-24.530386,1.702318,1.384797,1.262752,200,54.625083
6,Daily_Top_75pct,30.448644,-28.207802,1.647710,1.404567,1.079441,210,78.659589
8,Daily_Top_30pct,24.200552,-22.621518,1.536574,1.205041,1.069802,195,33.271400
5,Quantile_Q75,4.622386,-24.792343,1.282281,0.411409,0.186444,119,24.897147


KABUL TABLOSU


,Target,Model,Filter_Name,CAGR_%,Max_Drawdown_%,Profit_Factor,Sharpe,Calmar,Trade_Count,Acceptance_Count,ML_Accepted
0,Big_Winner_Label_2R,LogisticRegression,Quantile_Q40,38.377131,-17.322586,2.052734,1.860019,2.215439,152,5,True
1,Big_Winner_Label_2R,LogisticRegression,Quantile_Q25,50.996917,-24.454761,2.129541,2.151742,2.085357,185,5,True
2,Big_Winner_Label_2R,LogisticRegression,Quantile_Q50,33.449836,-19.503247,1.916607,1.791944,1.715091,147,4,False
3,Big_Winner_Label_2R,LogisticRegression,Baseline_Robot,36.064262,-23.126455,1.687332,1.575621,1.559438,214,3,False
4,Big_Winner_Label_2R,LogisticRegression,Quantile_Q60,28.877885,-19.494472,1.855069,1.630278,1.481337,142,3,False
5,Big_Winner_Label_2R,LogisticRegression,Daily_Top_50pct,30.975798,-24.530386,1.702318,1.384797,1.262752,200,3,False
6,Big_Winner_Label_2R,LogisticRegression,Daily_Top_30pct,24.200552,-22.621518,1.536574,1.205041,1.069802,195,2,False
7,Big_Winner_Label_2R,LogisticRegression,Quantile_Q75,4.622386,-24.792343,1.282281,0.411409,0.186444,119,2,False
8,Big_Winner_Label_2R,LogisticRegression,Daily_Top_75pct,30.448644,-28.207802,1.647710,1.404567,1.079441,210,1,False


## 4. Hedefler arasında tek Validation kararı


In [5]:
combined_acceptance = combine_acceptance_results(
    acceptance_frames
)

global_champion, ml_accepted = (
    select_global_validation_champion(
        combined_acceptance
    )
)

display(
    combined_acceptance[
        [
            "Target",
            "Model",
            "Filter_Name",
            "CAGR_%",
            "Max_Drawdown_%",
            "Profit_Factor",
            "Sharpe",
            "Calmar",
            "Trade_Count",
            "Signal_Pass_Rate_%",
            "Acceptance_Count",
            "ML_Accepted",
        ]
    ].sort_values(
        [
            "ML_Accepted",
            "Calmar",
            "CAGR_%",
        ],
        ascending=[False, False, False],
    )
)

print("Alternatif hedef ML kabul edildi mi?:", ml_accepted)
print("Global Validation champion:", global_champion)


,Target,Model,Filter_Name,CAGR_%,Max_Drawdown_%,Profit_Factor,Sharpe,Calmar,Trade_Count,Signal_Pass_Rate_%,Acceptance_Count,ML_Accepted
9,Big_Winner_Label_2R,LogisticRegression,Quantile_Q40,38.377131,-17.322586,2.052734,1.860019,2.215439,152,59.747843,5,True
10,Big_Winner_Label_2R,LogisticRegression,Quantile_Q25,50.996917,-24.454761,2.129541,2.151742,2.085357,185,74.678169,5,True
0,Meta_Label_1R,RandomForest,Quantile_Q40,39.909083,-22.234474,1.969850,1.933547,1.794919,172,59.747843,5,True
1,Meta_Label_1R,RandomForest,Quantile_Q25,42.105887,-25.482524,1.943877,1.947274,1.652344,180,74.678169,5,True
2,Meta_Label_1R,RandomForest,Quantile_Q60,32.569589,-17.511336,2.036495,1.681104,1.859915,151,39.827472,4,False
3,Meta_Label_1R,RandomForest,Daily_Top_30pct,32.327644,-18.266293,1.721245,1.520571,1.769798,201,33.271400,4,False
11,Big_Winner_Label_2R,LogisticRegression,Quantile_Q50,33.449836,-19.503247,1.916607,1.791944,1.715091,147,49.781022,4,False
4,Meta_Label_1R,RandomForest,Baseline_Robot,36.064262,-23.126455,1.687332,1.575621,1.559438,214,100.000000,3,False
12,Big_Winner_Label_2R,LogisticRegression,Baseline_Robot,36.064262,-23.126455,1.687332,1.575621,1.559438,214,100.000000,3,False
13,Big_Winner_Label_2R,LogisticRegression,Quantile_Q60,28.877885,-19.494472,1.855069,1.630278,1.481337,142,39.827472,3,False


Alternatif hedef ML kabul edildi mi?: True
Global Validation champion: {'Start_Value': 500000.0, 'End_Value': 956135.2465425462, 'Total_Return_%': 91.22704930850925, 'CAGR_%': 38.377130886722945, 'Max_Drawdown_%': -17.322585910397592, 'Profit_Factor': 2.0527341783527913, 'Win_Rate_%': 38.15789473684211, 'Expectancy_%': 4.394887466764602, 'Sharpe': 1.8600193283706028, 'Sortino': 2.679278984880014, 'Calmar': 2.2154389122519933, 'Trade_Count': 152, 'Outlier_Count': 0, 'Exposure_%': 94.97991967871486, 'Average_Open_Positions': 4.917670682730924, 'Max_Open_Positions': 6, 'Average_Invested_%': 58.1748447167567, 'Filter_Name': 'Quantile_Q40', 'Probability_Threshold': 0.511306, 'Keep_Top_Fraction': nan, 'Original_AL_Rows': 7535, 'Passed_AL_Rows': 4502, 'Signal_Pass_Rate_%': 59.74784339747843, 'Period_Start': Timestamp('2023-01-01 00:00:00'), 'Period_End': Timestamp('2024-12-31 00:00:00'), 'CAGR_%_Difference': 2.312868609566877, 'Max_Drawdown_%_Difference': 5.803869571800313, 'Profit_Factor_Dif

ML'nin kabul edilmesi için filtre şu koşulların tamamını sağlamalıdır:

- Baseline işlemlerin en az %50'sini korumalı.
- Validation CAGR yükselmeli.
- Validation Calmar yükselmeli.
- Profit Factor düşmemeli.
- Drawdown en fazla 3 yüzde puan kötüleşmeli.

Hiçbir aday kabul edilmezse Audit'te yalnızca Baseline Robot raporlanır.


## 5. Validation kararı sonrası Audit raporu


In [6]:
audit_filters = [
    MLFilterConfig(
        name="Baseline_Robot",
    )
]

audit_model = None
selected_target = None
selected_model_name = None
selected_filter_name = "Baseline_Robot"
selected_filter_config = None

if ml_accepted:
    selected_target = str(
        global_champion["Target"]
    )
    selected_model_name = str(
        global_champion["Model"]
    )
    selected_filter_name = str(
        global_champion["Filter_Name"]
    )

    development_validation = (
        events.loc[
            events["Period"].isin(
                ["Development", "Validation"]
            )
        ]
        .sort_values(["Signal_Date", "Ticker"])
        .reset_index(drop=True)
    )

    audit_events = (
        events.loc[
            events["Period"].eq(
                "Audit_2025_Plus"
            )
        ]
        .sort_values(["Signal_Date", "Ticker"])
        .reset_index(drop=True)
    )

    selected_templates = {
        selected_model_name: candidate_model_templates[
            selected_model_name
        ]
    }

    (
        audit_fitted_models,
        audit_event_metrics,
        audit_event_predictions,
    ) = train_on_development_predict_validation(
        development_data=development_validation,
        validation_data=audit_events,
        feature_columns=BASE_FEATURE_COLUMNS,
        target_column=selected_target,
        models=selected_templates,
        cutoff_date=AUDIT_START,
        embargo_days=5,
    )

    audit_model = audit_fitted_models[
        selected_model_name
    ]

    selected_filter_config = next(
        config
        for config in target_artifacts[
            selected_target
        ]["filter_grid"]
        if config.name == selected_filter_name
    )

    audit_filters.append(
        selected_filter_config
    )

    display(audit_event_metrics)

if ml_accepted:
    audit_probability_prices = add_model_probabilities(
        featured_prices=featured_prices,
        fitted_model=audit_model,
        feature_columns=BASE_FEATURE_COLUMNS,
        start=AUDIT_START,
        end=AUDIT_END,
    )
else:
    audit_probability_prices = featured_prices.copy()
    audit_probability_prices[
        "ML_Probability"
    ] = np.nan

audit_results, audit_outputs = run_filter_grid(
    probability_prices=audit_probability_prices,
    filters=audit_filters,
    strategy_config=FINAL_STRATEGY_CONFIG,
    portfolio_config=FINAL_PORTFOLIO_CONFIG,
    start=AUDIT_START,
    end=AUDIT_END,
)

audit_results = add_baseline_differences(
    audit_results
)

display(
    audit_results[
        [
            "Filter_Name",
            "CAGR_%",
            "Max_Drawdown_%",
            "Profit_Factor",
            "Sharpe",
            "Calmar",
            "Trade_Count",
            "Signal_Pass_Rate_%",
        ]
    ]
)


,Positive_Rate_%,Predicted_Positive_Rate_%,PR_AUC,Brier,Log_Loss,Precision_0_50,Recall_0_50,F1_0_50,ROC_AUC,Model,Train_Count,Validation_Count,Train_Positive_Rate_%
0,13.548387,39.569892,0.145414,0.240924,0.681223,0.141304,0.412698,0.210526,0.509496,LogisticRegression,2915,930,21.02916


,Filter_Name,CAGR_%,Max_Drawdown_%,Profit_Factor,Sharpe,Calmar,Trade_Count,Signal_Pass_Rate_%
0,Baseline_Robot,55.047940,-10.932331,2.054227,2.269298,5.035334,166,100.000000
1,Quantile_Q40,54.165166,-8.895519,2.451986,2.442341,6.089039,122,37.595489


## 6. Audit: Robot, kabul edilen ML ve BIST100


In [7]:
audit_comparison_rows = []

for filter_name, output in audit_outputs.items():
    metrics = portfolio_metrics(
        output["equity"],
        output["trades"],
    )
    metrics["Portfolio"] = filter_name
    audit_comparison_rows.append(metrics)

baseline_equity = audit_outputs[
    "Baseline_Robot"
]["equity"]

bist100_equity = build_benchmark_equity(
    market_prices=market_prices,
    comparison_dates=baseline_equity["Date"],
    initial_capital=(
        FINAL_PORTFOLIO_CONFIG.initial_capital
    ),
    include_costs=False,
    benchmark_name="BIST100",
)

bist100_metrics = portfolio_metrics(
    bist100_equity,
    pd.DataFrame(columns=["Return"]),
)
bist100_metrics["Portfolio"] = "BIST100 Gross"
audit_comparison_rows.append(bist100_metrics)

audit_comparison = pd.DataFrame(
    audit_comparison_rows
)

display(
    audit_comparison[
        [
            "Portfolio",
            "End_Value",
            "Total_Return_%",
            "CAGR_%",
            "Max_Drawdown_%",
            "Sharpe",
            "Calmar",
            "Profit_Factor",
            "Trade_Count",
        ]
    ]
)


,Portfolio,End_Value,Total_Return_%,CAGR_%,Max_Drawdown_%,Sharpe,Calmar,Profit_Factor,Trade_Count
0,Baseline_Robot,987738.701582,97.547740,55.047940,-10.932331,2.269298,5.035334,2.054227,166
1,Quantile_Q40,979022.359733,95.804472,54.165166,-8.895519,2.442341,6.089039,2.451986,122
2,BIST100 Gross,706623.382391,41.324676,24.958964,-17.061152,1.037176,1.462912,NaN,0


## 7. Sonuçları ve kararı kaydet


In [8]:
ML_RESULTS_DIR = (
    PROJECT_ROOT
    / "results"
    / "ml"
)
ML_RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

MODEL_DIR = PROJECT_ROOT / "models"
MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

combined_acceptance.to_csv(
    ML_RESULTS_DIR
    / "alternative_targets_validation_acceptance.csv",
    index=False,
)

audit_results.to_csv(
    ML_RESULTS_DIR
    / "alternative_targets_audit_results.csv",
    index=False,
)

audit_comparison.to_csv(
    ML_RESULTS_DIR
    / "alternative_targets_audit_vs_bist100.csv",
    index=False,
)

for target_column, artifacts in target_artifacts.items():
    safe_target = target_column.lower()

    artifacts["cv_fold_metrics"].to_csv(
        ML_RESULTS_DIR
        / f"{safe_target}_cv_fold_metrics.csv",
        index=False,
    )

    artifacts["cv_lift_summary"].to_csv(
        ML_RESULTS_DIR
        / f"{safe_target}_cv_lift_summary.csv",
        index=False,
    )

    artifacts[
        "validation_event_metrics"
    ].to_csv(
        ML_RESULTS_DIR
        / f"{safe_target}_validation_event_metrics.csv",
        index=False,
    )

    artifacts["event_thresholds"].to_csv(
        ML_RESULTS_DIR
        / f"{safe_target}_event_thresholds.csv",
        index=False,
    )

    artifacts[
        "validation_portfolio_results"
    ].to_csv(
        ML_RESULTS_DIR
        / f"{safe_target}_validation_portfolio.csv",
        index=False,
    )

decision = {
    "alternative_targets_tested": TARGET_COLUMNS,
    "ml_accepted_on_validation": bool(ml_accepted),
    "selected_target": selected_target,
    "selected_model": selected_model_name,
    "selected_filter": selected_filter_name,
    "final_system": (
        "Robot_with_alternative_target_ML"
        if ml_accepted
        else "Baseline_Robot"
    ),
    "validation_period": {
        "start": VALIDATION_START,
        "end": VALIDATION_END,
    },
    "audit_period": {
        "start": AUDIT_START,
        "end": AUDIT_END,
    },
}

DECISION_PATH = (
    MODEL_DIR
    / "alternative_target_ml_decision.json"
)

with DECISION_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        decision,
        file,
        ensure_ascii=False,
        indent=2,
    )

if ml_accepted and audit_model is not None:
    dump(
        audit_model,
        MODEL_DIR
        / "alternative_target_ml_model.joblib",
    )

print("Karar:", decision)
print("Karar dosyası:", DECISION_PATH)


Karar: {'alternative_targets_tested': ['Meta_Label_1R', 'Big_Winner_Label_2R'], 'ml_accepted_on_validation': True, 'selected_target': 'Big_Winner_Label_2R', 'selected_model': 'LogisticRegression', 'selected_filter': 'Quantile_Q40', 'final_system': 'Robot_with_alternative_target_ML', 'validation_period': {'start': '2023-01-01', 'end': '2024-12-31'}, 'audit_period': {'start': '2025-01-01', 'end': '2026-07-23'}}
Karar dosyası: c:\Users\okand\Desktop\Projects\Algorithmic Trading\BIST-Algo-Trade\models\alternative_target_ml_decision.json


## Karar ilkesi

- Validation'da hiçbir alternatif hedef kabul edilmezse ML araştırması burada
  durdurulur ve günlük sistem Baseline Robot olarak devam eder.
- Validation'da kabul edilip Audit'te belirgin biçimde bozulursa ML canlı
  sisteme alınmaz.
- Validation ve Audit olumlu olsa bile gerçek yeni out-of-sample doğrulama
  paper trading kayıtlarıdır.
